In [ ]:
import csv
import os
import numpy as np
import sentencepiece as spm
from google.colab import drive

drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/GenAI-Dataset (1)"
os.makedirs(drive_dir, exist_ok=True)
train_path = os.path.join(drive_dir, "train.tsv")

ANS_OPEN = "<ans>"
ANS_CLOSE = "</ans>"

train_pairs = []
with open(train_path, "r", encoding="utf-8") as f:
  reader = csv.reader(
      f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\"
  )
  for row in reader:
    if len(row) == 2:
      train_pairs.append((row[0], row[1]))
train_pairs = list(dict.fromkeys(train_pairs))

print(f"Successfully loaded {len(train_pairs)} pairs from Drive: {train_path}")

corpus_file = os.path.join(drive_dir, "sp_corpus.txt")
with open(corpus_file, "w", encoding="utf-8") as f:
  for src, tgt in train_pairs:
    f.write(src + "\n" + tgt + "\n")

print(f"Corpus prepared: {corpus_file}")

print("Training SentencePiece Tokenizer...")
spm.SentencePieceTrainer.train(
    input=corpus_file,
    model_prefix=os.path.join(drive_dir, "ur_sp"),
    vocab_size=8000,
    model_type="unigram",
    character_coverage=1.0,
    user_defined_symbols=[ANS_OPEN, ANS_CLOSE],  
    pad_id=0,  # Reserve <pad>
    unk_id=1,  # Reserve <unk>
    bos_id=2,  # Reserve <s>
    eos_id=3,  # Reserve </s>
)

print("Training complete! Files created: ur_sp.model, ur_sp.vocab")

sp = spm.SentencePieceProcessor(model_file=os.path.join(drive_dir, "ur_sp.model"))
PAD, UNK, BOS, EOS = 0, 1, 2, 3


for i in range(5):
  src, tgt = train_pairs[i]
  tgt_pieces = sp.encode(tgt, out_type=str)
  tgt_ids = sp.encode(tgt)
  round_trip_success = sp.decode(tgt_ids) == tgt

  print(f"\n--- Example {i+1} ---")
  print("Target Question:  ", tgt)
  print("Subword Pieces:   ", tgt_pieces)
  print("Token IDs:        ", tgt_ids)
  print("Round-trip Match: ", round_trip_success)

source_pieces = sum(len(sp.encode(src)) for src, _ in train_pairs)
target_pieces = sum(len(sp.encode(tgt)) for _, tgt in train_pairs)
source_words = sum(len(src.split()) for src, _ in train_pairs)
target_words = sum(len(tgt.split()) for _, tgt in train_pairs)
target_lengths = [len(sp.encode(tgt)) + 1 for _, tgt in train_pairs]  # Include EOS.
stats = {
    "source_fertility": source_pieces / source_words,
    "target_fertility": target_pieces / target_words,
    "target_p95": np.percentile(target_lengths, 95),
    "target_p99": np.percentile(target_lengths, 99),
    "target_max": max(target_lengths),
}
for name, value in stats.items():
  print(f"{name}: {value:.2f}")

with open(os.path.join(drive_dir, "tokenizer_stats.tsv"), "w", encoding="utf-8") as f:
  for name, value in stats.items():
    f.write(f"{name}\t{value}\n")

print("Tokenizer files and statistics saved to Google Drive.")

